In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import random
import warnings
warnings.filterwarnings("ignore")

In [2]:
seed=39
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

X_train=pd.read_csv('X_final_train.csv')
y_train=pd.read_csv('y_final_train.csv')
X_test=pd.read_csv('X_final_test.csv')
y_test=pd.read_csv('y_final_test.csv')
X_val=pd.read_csv('X_final_val.csv')
y_val=pd.read_csv('y_final_val.csv')

# sampled_indices=X_train.sample(n=3000, random_state=39).index
# X_train=X_train.loc[sampled_indices]
# y_train=y_train.loc[sampled_indices]
# sampled_indices1= X_test.sample(n=200, random_state=39).index
# X_test=X_test.loc[sampled_indices1]
# y_test=y_test.loc[sampled_indices1]
# sampled_indices2=X_val.sample(n=200, random_state=39).index
# X_val=X_val.loc[sampled_indices2]
# y_val=y_val.loc[sampled_indices2]


with open('features.txt', 'r') as f:
    feature_names = [line.strip() for line in f.readlines()]  

print(f"Total features in features.txt: {len(feature_names)}")
print(f"Columns in X_train: {X_train.shape[1]}")

if len(feature_names)==X_train.shape[1]:
    X_train.columns=feature_names
    X_val.columns=feature_names
    X_test.columns=feature_names

print(y_train.value_counts())

scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_val_scaled=scaler.transform(X_val)
X_test_scaled=scaler.transform(X_test)

pca=PCA(n_components=25, random_state=39)
X_train_scaled= pca.fit_transform(X_train_scaled)
X_val_scaled=pca.transform(X_val_scaled)
X_test_scaled=pca.transform(X_test_scaled)

label_encoder=LabelEncoder()
y_train_encoded=label_encoder.fit_transform(y_train)
y_val_encoded=label_encoder.transform(y_val)
y_test_encoded=label_encoder.transform(y_test)

X_train_tensor=torch.FloatTensor(X_train_scaled)
y_train_tensor=torch.LongTensor(y_train_encoded)
X_val_tensor=torch.FloatTensor(X_val_scaled)
y_val_tensor=torch.LongTensor(y_val_encoded)
X_test_tensor=torch.FloatTensor(X_test_scaled)
y_test_tensor=torch.LongTensor(y_test_encoded)

print(X_test_tensor.shape)
print(y_test_tensor.shape)

batch_size=256
train_dataset=TensorDataset(X_train_tensor,y_train_tensor)
train_loader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
val_dataset=TensorDataset(X_val_tensor,y_val_tensor)
val_loader=DataLoader(val_dataset,batch_size=batch_size)
test_dataset=TensorDataset(X_test_tensor,y_test_tensor)
test_loader=DataLoader(test_dataset,batch_size=batch_size)

Total features in features.txt: 561
Columns in X_train: 561
0 
5     1423
6     1413
4     1293
1     1226
2     1073
11     990
3      987
9      975
10     960
12     957
7      947
8      923
Name: count, dtype: int64
torch.Size([1581, 25])
torch.Size([1581])


In [3]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)

(13167, 561)
(13167, 1)
(1581, 561)


In [4]:

class MLP(nn.Module):
    def __init__(self,input_dim,hidden_units,dropout_rate,l1_reg,hidden_layers): #l1_reg is the lambda, the hyperparameter, since lambda in python is already used
        super(MLP,self).__init__()
        self.layers=nn.Sequential(
            nn.Linear(input_dim,hidden_units),
            nn.LeakyReLU(),
            nn.Dropout(dropout_rate),
        )
        for k in range(hidden_layers):
            self.layers.add_module(
                f"hidden_layer_{k}",
                nn.Sequential(
                    nn.Linear(hidden_units, hidden_units),
                    nn.LeakyReLU(),
                    nn.Dropout(dropout_rate)
            )
        )
        self.layers.add_module("output",nn.Linear(hidden_units,12)) 
        #The argmax activation function is used individually, since the CELoss calculation includes it already
        self.l1_reg=l1_reg

    def forward(self, x):
        return self.layers(x)

    def l1_loss(self):
        l1=0
        for param in self.parameters():
            l1+=torch.norm(param,p=1)
        return self.l1_reg*l1 
    #l1_reg is the lambda, the hyperparameter, since lambda in python is already used

In [5]:
def train_model(params,train_loader,val_loader,input_dim,test=False):
    model=MLP(input_dim=input_dim,hidden_units=params["hidden_units"],dropout_rate=params["dropout_rate"],l1_reg=params["l1_reg"],hidden_layers=params["hidden_layers"])
    optimizer=torch.optim.Adam(model.parameters(),lr=params["learning_rate"])
    best_val_loss=float('inf')
    patience=50
    trigger_times=0
    
    for epoch in range(params["epochs"]):
        model.train()
        train_loss=0.0
        train_correct=0
        train_total=0
        for batch_x,batch_y in train_loader:
            optimizer.zero_grad()
            outputs=model(batch_x)
            loss=nn.CrossEntropyLoss()(outputs,batch_y)+model.l1_loss() # hyperparameter lambda already multiplied witnin l1_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss+=loss.item()
            probs = torch.softmax(outputs, dim=1)
            predicted = torch.argmax(probs, dim=1)
            train_total += batch_y.size(0)
            train_correct += (predicted == batch_y).sum().item()
        train_acc = 100*train_correct/train_total  

        if(test==False):
            model.eval()
            val_loss=0.0
            correct=0
            total=0
            with torch.no_grad():
                for batch_x,batch_y in val_loader:
                    outputs=model(batch_x)
                    loss=nn.CrossEntropyLoss()(outputs,batch_y)+model.l1_loss() # hyperparameter lambda already multiplied witnin l1_loss
                    val_loss+=loss.item()
                    probs=torch.softmax(outputs,dim=1)
                    predicted=torch.argmax(probs,dim=1)
                    total+=batch_y.size(0)
                    correct+=(predicted==batch_y).sum().item()
        
        if (epoch+1)%25==0 or epoch==0:
            if(test==False):
                print(f'Epoch [{epoch+1}/{params["epochs"]}] - '
                    f'Train Loss: {train_loss/len(train_loader):.4f}, '
                    f'Train Acc: {train_acc:.2f}%, '
                    f'Val Loss: {val_loss/len(val_loader):.4f}, '
                    f'Val Acc: {100*correct/total:.2f}%')
            elif(test==True):
                    print(f'Epoch [{epoch+1}/{params["epochs"]}] - '
                    f'Train Loss: {train_loss/len(train_loader):.4f}, '
                    f'Train Acc: {train_acc:.2f}%')

        
        #Early stopping, the effect is not good
        # if val_loss<best_val_loss:
        #     best_val_loss=val_loss
        #     trigger_times=0
        # else:
        #     trigger_times+=1
        #     if trigger_times>=patience:
        #         print(f'Early stopping at epoch {epoch+1}')
        #         break
    if(test==False):
        return model,100*correct/total
    else:
        return model

In [6]:
# Use grid search to find the best hyperparameter combo
param_grid={"hidden_units":[10,50,100],"dropout_rate":[0,0.1,0.3],"l1_reg":[0,0.0001,0.001,0.1],"learning_rate":[1e-3,5e-4,1e-4],"epochs":[200],"hidden_layers":[3,5,10,20]}
all_params=[
    {
        "hidden_units":units,
        "dropout_rate":rate,
        "l1_reg":reg,
        "learning_rate":lr,
        "epochs":eps,
        "hidden_layers":layers
    }
    for units in param_grid["hidden_units"]
    for rate in param_grid["dropout_rate"]
    for reg in param_grid["l1_reg"]
    for lr in param_grid["learning_rate"]
    for eps in param_grid["epochs"]
    for layers in param_grid["hidden_layers"]
]

best_acc=0
best_params=None
best_model=None
input_dim=X_train_scaled.shape[1]
for params in all_params:
    print(f"\nTesting params: {params}")
    model,val_acc=train_model(params,train_loader,val_loader,input_dim)
    if val_acc>best_acc:
        best_acc=val_acc
        best_params=params
        best_model=model
    print(f"Validation Accuracy: {val_acc:.2f}%")

print(f"\nBest Params: {best_params}")
print(f"Best Validation Accuracy: {best_acc:.2f}%")


Testing params: {'hidden_units': 10, 'dropout_rate': 0, 'l1_reg': 0, 'learning_rate': 0.001, 'epochs': 200, 'hidden_layers': 3}
Epoch [1/200] - Train Loss: 2.4311, Train Acc: 12.33%, Val Loss: 2.1978, Val Acc: 17.20%
Epoch [25/200] - Train Loss: 0.2629, Train Acc: 89.23%, Val Loss: 0.3992, Val Acc: 83.18%
Epoch [50/200] - Train Loss: 0.2060, Train Acc: 91.77%, Val Loss: 0.4210, Val Acc: 83.87%
Epoch [75/200] - Train Loss: 0.1820, Train Acc: 92.55%, Val Loss: 0.4926, Val Acc: 84.06%
Epoch [100/200] - Train Loss: 0.1662, Train Acc: 93.20%, Val Loss: 0.5319, Val Acc: 84.12%
Epoch [125/200] - Train Loss: 0.1550, Train Acc: 93.51%, Val Loss: 0.5629, Val Acc: 84.44%
Epoch [150/200] - Train Loss: 0.1464, Train Acc: 93.98%, Val Loss: 0.6255, Val Acc: 83.43%
Epoch [175/200] - Train Loss: 0.1455, Train Acc: 93.90%, Val Loss: 0.6316, Val Acc: 84.25%
Epoch [200/200] - Train Loss: 0.1408, Train Acc: 94.08%, Val Loss: 0.6468, Val Acc: 84.31%
Validation Accuracy: 84.31%

Testing params: {'hidden_uni

In [ ]:
final_params=best_params.copy()
final_params["epochs"]=500
full_dataset=TensorDataset(torch.cat([X_train_tensor,X_val_tensor],0),torch.cat([y_train_tensor,y_val_tensor],0))
full_loader=DataLoader(full_dataset,batch_size=batch_size,shuffle=True)
final_model=train_model(final_params,full_loader,val_loader,input_dim,test=True)
final_model.eval()
test_loss=0.0
correct=0
total=0
with torch.no_grad():
    for batch_x,batch_y in test_loader:
        outputs=final_model(batch_x)
        loss=nn.CrossEntropyLoss()(outputs,batch_y)+final_model.l1_loss()
        test_loss+=loss.item()
        probs=torch.softmax(outputs,dim=1)
        predicted=torch.argmax(probs,dim=1)
        total+=batch_y.size(0)
        correct+=(predicted==batch_y).sum().item()
print(f'Final Test Accuracy: {100*correct/total:.2f}%')
#torch.save(final_model.state_dict(),"best_model.pth")

Epoch [1/500] - Train Loss: 2.1270, Train Acc: 19.79%
Epoch [25/500] - Train Loss: 0.3460, Train Acc: 91.25%
Epoch [50/500] - Train Loss: 0.2805, Train Acc: 93.36%
Epoch [75/500] - Train Loss: 0.2584, Train Acc: 93.86%
Epoch [100/500] - Train Loss: 0.2474, Train Acc: 94.24%
Epoch [125/500] - Train Loss: 0.2338, Train Acc: 94.66%
Epoch [150/500] - Train Loss: 0.2257, Train Acc: 94.90%
Epoch [175/500] - Train Loss: 0.2209, Train Acc: 95.05%
Epoch [200/500] - Train Loss: 0.2113, Train Acc: 95.31%
Epoch [225/500] - Train Loss: 0.2082, Train Acc: 95.38%
Epoch [250/500] - Train Loss: 0.2096, Train Acc: 95.52%
Epoch [275/500] - Train Loss: 0.2033, Train Acc: 95.50%
Epoch [300/500] - Train Loss: 0.2038, Train Acc: 95.63%
Epoch [325/500] - Train Loss: 0.2020, Train Acc: 95.78%
Epoch [350/500] - Train Loss: 0.1997, Train Acc: 95.82%
Epoch [375/500] - Train Loss: 0.1895, Train Acc: 96.22%
Epoch [400/500] - Train Loss: 0.1979, Train Acc: 96.11%
Epoch [425/500] - Train Loss: 0.1931, Train Acc: 95.9